# 00 -- Data setup and exploration
Audits the G2F 2024/2025 GxE Prediction Competition training data: confirms every
expected file actually contains data (not a failed/HTML download), profiles each
file's shape and columns, and cross-checks environment (year x location) codes
across the trait, meta, soil, weather, and EC files so join keys are understood
before any effect-alone model is built.

No modeling in this notebook. Execution only -- if this needs to run again
after real column names are confirmed against `readme.txt`, extend the checks
below rather than rewriting them.

## Environment setup (Colab or local)

In [ ]:
from pathlib import Path
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/g2f_effect_decomposition')
else:
    # Local run (VSCode). Data and results live on Drive; point this at wherever
    # Drive is mounted -- e.g. Google Drive for Desktop on WSL2 is typically
    # under /mnt/g/My Drive/... (adjust drive letter as needed).
    # G2F_BASE_PATH env var overrides this for testing without editing the notebook.
    BASE_PATH = Path(os.environ.get('G2F_BASE_PATH', '/mnt/g/My Drive/g2f_effect_decomposition'))

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

## Imports and config

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

DATA_DIR = BASE_PATH / 'data' / 'raw' / 'Training_data'

# Filenames as released in the CyVerse Training_data folder.
EXPECTED_FILES = {
    'trait':            '1_Training_Trait_Data_2014_2023.csv',
    'meta':             '2_Training_Meta_Data_2014_2023.csv',
    'soil':             '3_Training_Soil_Data_2015_2023.csv',
    'weather_full':     '4_Training_Weather_Data_2014_2023_full_year.csv',
    'weather_seasons':  '4_Training_Weather_Data_2014_2023_seasons_only.csv',
    'genotype_vcf':     '5_Genotype_Data_All_2014_2025_Hybrids.vcf',
    'genotype_numeric': '5_Genotype_Data_All_2014_2025_Hybrids_numerical.txt',
    'ec':               '6_Training_EC_Data_2014_2023.csv',
    'key_inbreds':      'key_inbreds_G2F_2014-2025.txt',
}

pd.set_option('display.max_columns', 50)

## File integrity check
CyVerse gates downloads behind a reCAPTCHA click-through in its browser app.
A file pulled with a plain HTTP client (rather than a real browser download)
can silently come back as the ~7KB Angular landing page instead of the actual
data. Every file gets checked for that before anything downstream trusts it.

In [ ]:
def integrity_status(path: Path) -> str:
    """Returns 'missing', 'suspect_html', or 'ok' for a downloaded file."""
    if not path.exists():
        return 'missing'
    with open(path, 'rb') as f:
        head = f.read(512)
    if b'<!DOCTYPE html' in head or b'<html' in head[:200]:
        return 'suspect_html'
    return 'ok'


status_rows = []
for key, filename in EXPECTED_FILES.items():
    path = DATA_DIR / filename
    status = integrity_status(path)
    size = path.stat().st_size if path.exists() else None
    status_rows.append({'key': key, 'file': filename, 'status': status, 'size_bytes': size})

status_df = pd.DataFrame(status_rows)
print(status_df.to_string(index=False))

bad = status_df[status_df['status'] != 'ok']
if len(bad):
    print(f"\n{len(bad)} file(s) failed the integrity check -- re-download these before"
          " trusting anything profiled below:")
    print(bad['file'].to_string(index=False))
else:
    print("\nAll files passed the integrity check.")

## Tabular file profiles
Loads a small sample of each CSV that passed the integrity check: shape,
columns, dtypes, and a head preview. Skips any file that failed the check
above rather than profiling garbage.

In [ ]:
TABULAR_KEYS = ['trait', 'meta', 'soil', 'weather_full', 'weather_seasons', 'ec']

tabular_frames: dict[str, pd.DataFrame] = {}

for key in TABULAR_KEYS:
    filename = EXPECTED_FILES[key]
    row = status_df[status_df['key'] == key].iloc[0]
    if row['status'] != 'ok':
        print(f"[skip] {key} ({filename}): {row['status']}")
        continue

    df_full = pd.read_csv(DATA_DIR / filename)
    tabular_frames[key] = df_full

    print(f"=== {key} ({filename}) ===")
    print(f"shape: {df_full.shape}")
    print(f"columns: {list(df_full.columns)}")
    print(df_full.head(3))
    print()

## Missing-value profile
Per column missing-value percentage for each loaded tabular file -- flags
which fields are usable as-is vs. need imputation or exclusion.

In [ ]:
for key, df_full in tabular_frames.items():
    miss = (df_full.isna().mean() * 100).round(1)
    miss = miss[miss > 0].sort_values(ascending=False)
    print(f"=== {key}: missing % by column ===")
    print(miss.to_string() if len(miss) else "(no missing values)")
    print()

## Environment (year x location) join-key audit
G2F's known misjoin risk is in the environment code used to link trait,
meta, soil, weather, and EC records. This looks for any column whose name
contains 'env' (case-insensitive) in each loaded file, then compares the
sets of values across files -- mismatches here mean the join key isn't as
simple as a direct string match and the readme needs to be checked before
building any loader.

In [ ]:
env_value_sets: dict[str, set] = {}

for key, df_full in tabular_frames.items():
    env_cols = [c for c in df_full.columns if 'env' in c.lower()]
    if not env_cols:
        print(f"{key}: no column with 'env' in its name -- columns are {list(df_full.columns)}")
        continue
    col = env_cols[0]
    env_value_sets[key] = set(df_full[col].astype(str).unique())
    print(f"{key}: using column '{col}' ({df_full[col].nunique()} unique values)"
          + (f" -- also found {env_cols[1:]}" if len(env_cols) > 1 else ""))

print()
keys = list(env_value_sets.keys())
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        a, b = keys[i], keys[j]
        overlap = env_value_sets[a] & env_value_sets[b]
        only_a = env_value_sets[a] - env_value_sets[b]
        only_b = env_value_sets[b] - env_value_sets[a]
        print(f"{a} vs {b}: {len(overlap)} shared, {len(only_a)} only in {a}, {len(only_b)} only in {b}")
        if only_a:
            print(f"  sample only in {a}: {sorted(only_a)[:5]}")
        if only_b:
            print(f"  sample only in {b}: {sorted(only_b)[:5]}")

## Genotype data
Profiles both genotype formats without loading the full VCF body into
memory -- header/sample/contig info only via `cyvcf2`, plus a shape check
on the numerical marker matrix and a preview of the key-inbreds list.

In [ ]:
geno_vcf_status = status_df[status_df['key'] == 'genotype_vcf'].iloc[0]['status']

if geno_vcf_status == 'ok':
    from cyvcf2 import VCF

    vcf_path = DATA_DIR / EXPECTED_FILES['genotype_vcf']
    vcf = VCF(str(vcf_path))
    print(f"VCF samples: {len(vcf.samples)}")
    print(f"VCF sample preview: {vcf.samples[:5]}")

    n_variants = 0
    contigs = set()
    for i, variant in enumerate(vcf):
        contigs.add(variant.CHROM)
        n_variants += 1
        if i >= 9999:  # cap the scan -- full-file variant count can be done separately if needed
            print("(stopped after 10,000 variants -- re-run without the cap for an exact count)")
            break
    print(f"variants scanned: {n_variants}")
    print(f"contigs seen: {sorted(contigs)}")
else:
    print(f"[skip] genotype_vcf: {geno_vcf_status}")

In [ ]:
geno_num_status = status_df[status_df['key'] == 'genotype_numeric'].iloc[0]['status']

if geno_num_status == 'ok':
    geno_num_path = DATA_DIR / EXPECTED_FILES['genotype_numeric']
    geno_num = pd.read_csv(geno_num_path, sep=None, engine='python', nrows=5)
    print(f"numerical genotype file -- preview shape: {geno_num.shape}")
    print(f"columns (first 10): {list(geno_num.columns[:10])}")
    print(geno_num.iloc[:, :6])
else:
    print(f"[skip] genotype_numeric: {geno_num_status}")

In [ ]:
key_inbreds_status = status_df[status_df['key'] == 'key_inbreds'].iloc[0]['status']

if key_inbreds_status == 'ok':
    key_inbreds_path = DATA_DIR / EXPECTED_FILES['key_inbreds']
    with open(key_inbreds_path) as f:
        lines = [line.strip() for line in f]
    print(f"key_inbreds: {len(lines)} lines")
    print(f"preview: {lines[:10]}")
else:
    print(f"[skip] key_inbreds: {key_inbreds_status}")

## Summary
Consolidated status -- what's confirmed usable vs. what still needs a
re-download or a readme check, so the next session knows exactly where to
pick up.

In [ ]:
print(status_df.to_string(index=False))
print()
print(f"Tabular files loaded: {list(tabular_frames.keys())}")
print(f"Env join-key sets compared: {list(env_value_sets.keys())}")